In [24]:
import argparse
import json
import logging
import os
import pathlib
import pprint
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import natsort
import numpy as np
import pandas as pd
import seaborn as sns
import skimage
import tifffile
import torch
from PIL import Image
from rich.pretty import pprint

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import logging

logging.getLogger("ultrack").setLevel(logging.INFO)

from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import napari
    import tqdm

In [25]:
start_time = time.time()

In [38]:
if not in_notebook:
    print("Running as script")
    # set up arg parser
    parser = argparse.ArgumentParser(description="Segment the nuclei of a tiff image")

    parser.add_argument(
        "--well_fov",
        type=str,
        help="Path to the input directory containing the tiff images",
    )

    parser.add_argument(
        "--plate_name",
        type=str,
        help="Name of the plate to process (e.g., 'Wave2')",
    )

    args = parser.parse_args()
    well_fov = args.well_fov
    plate_name = args.plate_name

else:
    print("Running in a notebook")
    well_fov = "B2_2"  # example well_fov
    plate_name = "plate_2"  # example plate name

image_base_dir = bandicoot_check(
    root_dir=root_dir,
    bandicoot_mount_path=pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(),
)

raw_image_input_dir = pathlib.Path(
    image_base_dir / "processed_data" / "0.renamed_files" / plate_name / well_fov
).resolve(strict=True)

segmentation_mask_input_dir = pathlib.Path(
    image_base_dir
    / "processed_data"
    / "2.cell_segmentation_masks"
    / plate_name
    / well_fov
).resolve()

track_save_path = pathlib.Path(
    image_base_dir / "processed_data" / "3.cell_tracks" / plate_name / well_fov
).resolve()


temporary_output_dir = pathlib.Path("../tmp_output").resolve()
figures_output_dir = pathlib.Path("../figures").resolve()
results_output_dir = pathlib.Path("../results").resolve()
track_save_path.mkdir(exist_ok=True, parents=True)
temporary_output_dir.mkdir(exist_ok=True, parents=True)
figures_output_dir.mkdir(exist_ok=True, parents=True)
results_output_dir.mkdir(exist_ok=True, parents=True)

Running in a notebook


In [39]:
file_extensions = {".tif", ".tiff"}
# get all the raw image files
raw_images = list(raw_image_input_dir.glob("*"))
raw_images = [f for f in raw_images if f.suffix in file_extensions]
raw_images = sorted(raw_images)

# get all the segmentation mask files
segmentation_masks = list(segmentation_mask_input_dir.glob("*"))
segmentation_masks = [f for f in segmentation_masks if f.suffix in file_extensions]
segmentation_masks = sorted(segmentation_masks)


nuclei_files = [f for f in raw_images if "C4" in f.name.split("_")[3]]
mask_files = [f for f in segmentation_masks if "nuclei" in f.name]

nuclei_files = natsort.natsorted(nuclei_files)
mask_files = natsort.natsorted(mask_files)

print(f"Found {len(mask_files)} segmentation mask files in the input directory")
print(f"Found {len(nuclei_files)} nuclei files in the input directory")

Found 288 segmentation mask files in the input directory
Found 288 nuclei files in the input directory


In [40]:
# read in the masks and create labels
labels = []
for tiff_file in mask_files:
    img = tifffile.imread(tiff_file)
    labels.append(img)

labels = np.array(labels)

images = []
for tiff_file in nuclei_files:
    img = tifffile.imread(tiff_file)
    images.append(img)
images = np.array(images)

In [41]:
props_df_list = []
for t in tqdm.tqdm(range(labels.shape[0])):
    # get the regionprops of the labels
    props = skimage.measure.regionprops_table(
        labels[t],
        properties=[
            "label",
            "bbox",
            "centroid",
        ],
    )
    props_df = pd.DataFrame(props)
    props_df["t"] = t
    props_df_list.append(props_df)

props_df = pd.concat(props_df_list, ignore_index=True)
print(props_df.shape)
props_df.head()

  0%|          | 0/288 [00:00<?, ?it/s]

(150705, 8)


,label,bbox-0,bbox-1,bbox-2,bbox-3,centroid-0,centroid-1,t
0,1,0,142,33,208,13.918462,176.707047,0
1,2,0,421,57,489,24.898471,453.928083,0
2,3,0,659,27,702,11.704805,678.473684,0
3,4,0,753,39,821,16.491169,786.968019,0
4,5,0,929,53,998,23.892328,962.657228,0


In [42]:
props_df.tail()

,label,bbox-0,bbox-1,bbox-2,bbox-3,centroid-0,centroid-1,t
150700,594,1963,1551,1984,1571,1973.399390,1560.054878,287
150701,595,1969,581,1999,633,1984.449838,608.399676,287
150702,596,1969,889,1999,939,1985.265766,916.293694,287
150703,597,1970,356,1999,422,1986.139535,385.103101,287
150704,598,1979,446,1998,498,1989.948518,470.127925,287


In [47]:
# load the tracks
tracks_path = track_save_path / "cell_tracks.parquet"
df = pd.read_parquet(tracks_path)
print(df.shape)
df.drop(columns=["z"], inplace=True)
df["tracklet_id"] = df["tracklet_id"].astype(int)
df.head()

(710, 5)


,tracklet_id,t,y,x
0,1,0.0,924.598826,1992.585127
1,2,59.0,1995.517647,127.952941
2,3,82.0,483.907957,1284.972864
3,4,91.0,1967.626039,800.476454
4,5,95.0,457.380030,1221.940387


In [50]:
df["tracklet_id"].unique

<bound method Series.unique of 0        1
1        2
2        3
3        4
4        5
      ... 
705    706
706    707
707    708
708    709
709    710
Name: tracklet_id, Length: 710, dtype: int64>

In [45]:
print(df.shape)
print(props_df.shape)

(710, 4)
(150705, 8)


In [52]:
tmp_df = pd.merge(
    props_df,
    df,
    left_on=["centroid-0", "centroid-1", "t"],
    right_on=["y", "x", "t"],
    how="left",
).dropna()

tmp_df.loc[tmp_df["tracklet_id"] == 3.0]

,label,bbox-0,bbox-1,bbox-2,bbox-3,centroid-0,centroid-1,t,tracklet_id,y,x
34771,141,458,1254,511,1320,483.907957,1284.972864,82,3.0,483.907957,1284.972864
